# 08 — ML: Strike-Probability Model

Trains a called-strike probability model, registers it in **Unity Catalog
Models**, applies a champion/challenger alias, and scores all pitches back
into gold.

## Why this matters — Well-Architected Framework

| WAF pillar | What this notebook does, and why it's good |
|------------|---------------------------------------------|
| **Operational Excellence** | MLflow run → UC Model Registry → `models:/{name}@champion` alias. Production scoring jobs reference the *alias*, not a version number — so promoting a new model is a metadata update, not a code deploy. |
| **Data Governance** | Registered model lineage links the model back to `{G}.fact_pitches` automatically — you can answer "which model was trained on which table?" from SQL, not from a ticket thread. |
| **Reliability** | `infer_signature` + `input_example` = any downstream caller gets a schema-checked interface. MLflow refuses to serve a request whose payload doesn't match the signature. |
| **Security, Compliance & Privacy** | Registering under `{UC_CATALOG}.{GOLD_SCHEMA}.strike_probability` means the model is a UC asset: `GRANT EXECUTE` + tags + access logs all work the same as for a table. |

> **WAF framing for the MLB team:** this is a toy model (physics-only,
> no batter identity) — the *structure* is what matters. Swap the
> `GradientBoostingClassifier` for whatever — the MLOps surface area
> doesn't change.


In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG   = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA    = os.getenv("UC_SCHEMA",  "mlb_gumbo_waf")
GOLD_SCHEMA  = f"{UC_SCHEMA}_gold"
SILVER_SCHEMA= f"{UC_SCHEMA}_silver"

G = f"{UC_CATALOG}.{GOLD_SCHEMA}"
S = f"{UC_CATALOG}.{SILVER_SCHEMA}"

MODEL_NAME   = f"{UC_CATALOG}.{GOLD_SCHEMA}.strike_probability"
print(f"Model will be registered at: {MODEL_NAME}")


Model will be registered at: alexander_booth.mlb_gumbo_waf_gold.strike_probability


## 1. Training set

A *called strike* is a pitch that the batter did not swing at and was ruled a
strike. We build a tidy training frame with just the physical pitch features —
deliberately omitting the batter/pitcher identity so the model generalizes.

In [2]:
training_df = spark.sql(f"""
    SELECT
        CAST((call_code = 'C') AS INT) AS label_called_strike,
        start_speed_mph,
        spin_rate_rpm,
        plate_x,
        plate_z,
        sz_top,
        sz_bot,
        break_vertical_induced,
        break_horizontal,
        extension_ft,
        batter_side,
        pitcher_hand
    FROM {G}.fact_pitches
    WHERE call_code IN ('C', 'B')
      AND start_speed_mph IS NOT NULL
      AND plate_x IS NOT NULL AND plate_z IS NOT NULL
""").toPandas()

print(f"Rows: {len(training_df):,}")
print(f"Class balance: {training_df['label_called_strike'].mean():.3f} called-strike rate")
training_df.head()


Rows: 360,546
Class balance: 0.327 called-strike rate


,label_called_strike,start_speed_mph,spin_rate_rpm,plate_x,plate_z,sz_top,sz_bot,break_vertical_induced,break_horizontal,extension_ft,batter_side,pitcher_hand
0,1,93.9,2083.0,0.271667,2.567177,3.730001,1.920283,1.7,17.2,6.223607,L,R
1,0,81.7,2880.0,-0.830531,3.535838,3.684805,1.795321,-3.2,-19.2,6.171363,L,R
2,0,96.8,2257.0,1.012342,2.324632,3.730000,1.806753,12.7,9.0,6.464287,L,R
3,1,87.9,2650.0,-0.433371,3.259762,3.651513,1.755841,1.2,-0.7,6.448334,L,R
4,1,97.1,2214.0,-0.581542,1.693440,3.550002,1.754582,12.1,7.2,6.441446,L,R


## 2. Train + log with MLflow → UC model registry

In [3]:
import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, brier_score_loss

# Point MLflow at UC (Models in UC)
mlflow.set_registry_uri("databricks-uc")

X = training_df.drop(columns=["label_called_strike"])
y = training_df["label_called_strike"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ["start_speed_mph","spin_rate_rpm","plate_x","plate_z","sz_top","sz_bot",
            "break_vertical_induced","break_horizontal","extension_ft"]
cat_cols = ["batter_side","pitcher_hand"]

pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])
model = Pipeline([("pre", pre), ("clf", GradientBoostingClassifier(random_state=42, n_estimators=100, max_depth=3))])

with mlflow.start_run(run_name=f"strike_probability_{os.getenv('MLB_SEASON','2025')}") as run:
    model.fit(X_train, y_train)

    proba_test = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba_test)
    brier = brier_score_loss(y_test, proba_test)

    mlflow.log_metric("auc",   auc)
    mlflow.log_metric("brier", brier)
    mlflow.log_param("training_rows", len(X_train))
    mlflow.log_param("feature_set", "physics_only")
    mlflow.set_tag("waf_pillar", "operational_excellence")
    mlflow.set_tag("training_data", f"{G}.fact_pitches")

    signature = infer_signature(X_train.head(100), model.predict_proba(X_train.head(100))[:, 1])

    info = mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(3),
        registered_model_name=MODEL_NAME,
    )

    print(f"  AUC   : {auc:.4f}")
    print(f"  Brier : {brier:.4f}")
    print(f"  Model URI: {info.model_uri}")
    print(f"  Registered: {MODEL_NAME}")

version = info.registered_model_version
print(f"\nRegistered model version: {version}")


Registered model 'alexander_booth.mlb_gumbo_waf_gold.strike_probability' already exists. Creating a new version of this model...
2026/04/21 15:58:21 WARNING mlflow.store._unity_catalog.registry.rest_store: Unable to get model version source run's workspace ID from request headers. No run link will be recorded for the model version


/Users/alexander.booth/miniconda3/envs/demo-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Uploading artifacts:   0%|          | 0/7 [00:00<?, ?it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:02,  2.64it/s]

Uploading artifacts:  14%|█▍        | 1/7 [00:00<00:02,  2.64it/s]

...

Uploading artifacts:  86%|████████▌ | 6/7 [00:00<00:00,  8.90it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00,  8.90it/s]

Uploading artifacts: 100%|██████████| 7/7 [00:00<00:00, 10.87it/s]

  AUC   : 0.9847
  Brier : 0.0457
  Model URI: runs:/7ab4c4fba4154a77a5ba4812afc0c6bf/model
  Registered: alexander_booth.mlb_gumbo_waf_gold.strike_probability

Registered model version: 3


Created version '3' of model 'alexander_booth.mlb_gumbo_waf_gold.strike_probability'.


## 3. Champion / challenger alias

The alias is the WAF pattern for zero-downtime promotion — batch-scoring jobs
reference `models:/{MODEL_NAME}@champion` and promotion becomes a single
`set_registered_model_alias` call.

In [4]:
from mlflow.tracking import MlflowClient
client = MlflowClient()

# Set the new version as the challenger first. If no champion yet, make it champion too.
client.set_registered_model_alias(name=MODEL_NAME, alias="challenger", version=str(version))
try:
    _ = client.get_model_version_by_alias(MODEL_NAME, "champion")
    print(f"  Existing champion kept. New challenger = v{version}.")
except Exception:
    client.set_registered_model_alias(name=MODEL_NAME, alias="champion", version=str(version))
    print(f"  No champion yet — promoted v{version} to champion.")


  Existing champion kept. New challenger = v3.


## 4. Batch inference → gold.pitch_strike_probability

In [5]:
import mlflow.pyfunc
import pandas as pd

# Load the champion model by alias — this is the production pattern
champion = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")

# Score all eligible pitches
score_pdf = spark.sql(f"""
    SELECT
        pitch_sk, game_sk, season, official_date,
        start_speed_mph, spin_rate_rpm, plate_x, plate_z, sz_top, sz_bot,
        break_vertical_induced, break_horizontal, extension_ft,
        batter_side, pitcher_hand
    FROM {G}.fact_pitches
    WHERE start_speed_mph IS NOT NULL AND plate_x IS NOT NULL AND plate_z IS NOT NULL
""").toPandas()

features = score_pdf[num_cols + cat_cols]
score_pdf["called_strike_prob"] = champion.predict(features)
score_pdf = score_pdf[["pitch_sk","game_sk","season","official_date","called_strike_prob"]]

scored = spark.createDataFrame(score_pdf)
out_table = f"{G}.pitch_strike_probability"
(scored.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .clusterBy("season", "official_date")
    .saveAsTable(out_table))

spark.sql(f"""
    COMMENT ON TABLE {out_table} IS
    'Per-pitch called-strike probability predictions from the strike_probability UC model (champion alias). Refresh nightly.'
""")

print(f"Wrote: {out_table}")
print(f"  Rows: {spark.table(out_table).count():,}")
spark.sql(f"SELECT * FROM {out_table} LIMIT 5").show(truncate=False)


Wrote: alexander_booth.mlb_gumbo_waf_gold.pitch_strike_probability


  Rows: 724,002


+--------------------------------+--------------------------------+------+-------------+------------------+
|pitch_sk                        |game_sk                         |season|official_date|called_strike_prob|
+--------------------------------+--------------------------------+------+-------------+------------------+
|71882bc4f3339ceedf3b3adebaca04ed|e4f8dab326ad70e23b9e54819fe96621|2025  |2025-05-28   |1                 |
|dbd0f8edbb3704524cd04207e0638716|e4f8dab326ad70e23b9e54819fe96621|2025  |2025-05-28   |0                 |
|6d1d903c6d8dea725dd7a750f127abd8|e4f8dab326ad70e23b9e54819fe96621|2025  |2025-05-28   |0                 |
|0d7f411530119b17b87793360c6ba054|e4f8dab326ad70e23b9e54819fe96621|2025  |2025-05-28   |1                 |
|620b916fed6510ffc92a0bd6e1ae8621|e4f8dab326ad70e23b9e54819fe96621|2025  |2025-05-28   |1                 |
+--------------------------------+--------------------------------+------+-------------+------------------+

